[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghodsizadeh/tehran-stocks/blob/master/Example/Read_Data.ipynb)
[![Documentation](https://img.shields.io/badge/Docs-%D9%85%D8%B3%D8%AA%D9%86%D8%AF%D8%A7%D8%AA%20%D9%81%D8%A7%D8%B1%D8%B3%DB%8C-0d9488.svg)](https://ghodsizadeh.github.io/tehran-stocks/)
[![GitHub stars](https://img.shields.io/github/stars/ghodsizadeh/tehran-stocks?style=social)](https://github.com/ghodsizadeh/tehran-stocks)

# Tehran Stocks v3 — Quickstart & Market Data

Fetch live Tehran Stock Exchange (`cdn.tsetmc.com`) quotes, individual vs. institutional (`حقیقی / حقوقی`) flows, 5-level order books, disclosures, Jalali-indexed adjusted history, and optional local SQLite storage — with **zero import side effects**.

> **Note:** All TSETMC prices and trade values are in **Iranian Rials (IRR)** (`1 Toman = 10 IRR`; `1 Billion Toman = 1e10 IRR`).

## 1. Installation

Requires **Python 3.11+**. Importing `tehran_stocks` creates no files and no database unless you explicitly opt in.

In [ ]:
%pip install -q "tehran-stocks>=3,<4"

In [ ]:
import tehran_stocks as ts

print(f"tehran-stocks version: {ts.__version__}")

## 2. Search Symbols, Live Quote & Fundamentals

Use `with ts.Client() as client:` to reuse an HTTP/2 connection and the in-memory symbol cache across calls. Persian (`ک`, `ی`) and Arabic (`ك`, `ي`) characters are normalized automatically.

In [ ]:
SYMBOL = "کگل"  # Also accepts Arabic 'كگل', ISIN, or numeric insCode

with ts.Client() as client:
    matches = client.instruments.search("گل گهر")
    quote = client.prices.quote(SYMBOL)
    info = client.instruments.info(SYMBOL)

print("Top search matches:")
for item in matches[:5]:
    print(f"  {item.symbol:10s} | {item.name} | insCode={item.ins_code}")

print(f"\n=== {info.symbol} ({info.name}) ===")
print(f"Sector:        {info.sector_name} ({info.sector_code})")
print(f"State:         {quote.state_title} ({quote.state})")
print(f"Last price:    {quote.last:,.0f} IRR ({quote.change_pct:+.2f}%)")
print(f"Close price:   {quote.close:,.0f} IRR | Yesterday: {quote.yesterday:,.0f} IRR")
print(f"Volume:        {quote.volume:,} (Base volume: {info.base_volume:,})")
print(f"Trade value:   {quote.value / 1e10:,.2f} Billion Toman")
print(f"EPS / Sec P/E: {info.eps} / {info.sector_pe}")

## 3. Individual vs. Institutional Flow (`حقیقی / حقوقی`) & 5-Level Order Book

Pass `include_thresholds=True` to `order_book()` to enable `is_buy_queue` and `is_sell_queue` detection.

In [ ]:
with ts.Client() as client:
    flow = client.client_type.live(SYMBOL)
    flow_10d = client.client_type.history(SYMBOL, days=10)
    book = client.prices.order_book(SYMBOL, include_thresholds=True)

print(f"Real money flow (today): {flow.real_money_flow / 1e10:+,.2f} Billion Toman")
print(f"Per-capita buy:          {flow.buy_per_capita / 1e7:,.1f} Million Toman")
print(f"Per-capita sell:         {flow.sell_per_capita / 1e7:,.1f} Million Toman")
print(f"Buyer power:             {flow.buyer_power:.2f}")
print(f"10-day net real flow:    {sum(d.real_money_flow for d in flow_10d) / 1e10:+,.2f} Billion Toman")

print(f"\nBuy queue: {book.is_buy_queue} | Sell queue: {book.is_sell_queue} | Spread: {book.spread}")
book_df = ts.to_dataframe(book.levels, jalali=False, index_by_date=False)
book_df

## 4. Shareholders, Codal Filings, Supervisor Messages & ETF NAV

In [ ]:
with ts.Client() as client:
    holders = client.disclosures.shareholders(SYMBOL)
    filings = client.disclosures.codal(SYMBOL, limit=5)
    notices = client.disclosures.messages(SYMBOL, limit=5)
    etf_info = client.funds.etf("بمان")

print("--- Top Major Shareholders (>1%) ---")
for h in holders[:5]:
    print(f"  {h.name}: {h.percent:.2f}% (daily change: {h.change:+,})")

print("\n--- Recent Codal Disclosures ---")
for f in filings:
    print(f"  [{f.publish_datetime}] {f.title}")

print("\n--- Recent Supervisor Messages ---")
for m in notices:
    print(f"  [{m.date} {m.time}] {m.title}")

print(
    f"\nETF بمان — Redemption NAV: {etf_info.redemption_nav:,.0f} IRR | "
    f"Statistical NAV: {etf_info.statistical_nav:,.0f} IRR"
)

## 5. Adjusted Daily History with Jalali Calendar (`jalali-pandas`)

Always pass `adjusted=True` for returns or technical indicators so dividend and capital-increase gaps are backward-adjusted.

In [ ]:
with ts.Client() as client:
    bars = client.prices.history(SYMBOL, days=120, adjusted=True)

df = ts.to_dataframe(bars, jalali=True, index_by_date=True)
df[["open", "high", "low", "close", "volume", "value"]].tail(10)

In [ ]:
# Export to an Excel-compatible UTF-8 CSV (with BOM so Persian characters render cleanly)
df.to_csv("kgl_adjusted_120d.csv", encoding="utf-8-sig")
print(f"Saved {len(df)} rows to kgl_adjusted_120d.csv")

## 6. Optional Local SQLite Storage & Pre-Built `data-latest` Snapshot

When you need offline backtesting or SQL analytics, opt into `SQLiteStore` (zero extra dependencies) or download the full-market SQLite snapshot (~184 MiB compressed) from GitHub Releases:

```bash
curl -fL https://github.com/ghodsizadeh/tehran-stocks/releases/download/data-latest/tse.sqlite.zst -o tse.sqlite.zst
zstd -d tse.sqlite.zst -o tse.sqlite
tse db sync --db ./tse.sqlite --incremental --client-type
```

In [ ]:
from tehran_stocks.storage import SyncService, open_store

store = open_store("./sample_tse.sqlite")
try:
    result = SyncService(store, max_concurrency=4).sync_all(
        symbols=["کگل", "فولاد"],
        incremental=True,
        include_client_types=True,
    )
    print(f"Synced {result.prices} price rows and {result.client_types} client-type rows.")

    # Validated read-only SQL (blocks INSERT/UPDATE/DELETE/DROP)
    rows = store.query_readonly(
        """
        SELECT i.symbol, p.date, p.close, p.volume,
               ROUND((c.buy_i_value - c.sell_i_value) / 10000000000.0, 2) AS real_flow_bt
        FROM daily_prices p
        JOIN instruments i ON i.ins_code = p.ins_code
        LEFT JOIN client_types c ON c.ins_code = p.ins_code AND c.date = p.date
        WHERE i.symbol = :symbol
        ORDER BY p.date DESC
        LIMIT 10
        """,
        params={"symbol": "کگل"},
    )
finally:
    store.close()

ts.to_dataframe(rows, jalali=False, index_by_date=False)